In [1]:
import os
from dotenv import load_dotenv

In [2]:
env_path=r"C:\Users\Lenovo\OneDrive\Desktop\RAG Project\GOOGLE_API_KEY.env"
env=load_dotenv(dotenv_path=env_path)
api_key=os.getenv("GOOGLE_API_KEY")
if api_key:
    print("API key successfully loaded")
else:
    print("API key was not found")

API key successfully loaded


In [3]:
from langchain_google_genai import ChatGoogleGenerativeAI

In [5]:
import google.generativeai as genai
genai.configure(api_key=api_key)
models=[]
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        a=m.name
        models.append(a)
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-exp-1206
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-3-1b-it
models/gemma-3-4b-it
models/gemma-3-12b-it
models/gemma-3-27b-it
models/gemma-3n-e4b-it
models/gemma-3n-e2b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-2.5-flash-preview-09-2025
models/gemini-2.5-flash-lite-preview-09-2025
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3-pro-image-preview
models/nano-banana-pro-preview
models/gemini-robotics-er-1.5-preview
models/gemini-2.5-computer-use-preview-10-2025
models/deep-research-pro-preview-12-2025


In [6]:
model_clean=[]
for m in models:
    m=m.split('/')[1]
    model_clean.append(m)

In [7]:
working_models=[]

In [ ]:
for m in model_clean:
    try:
        llm=ChatGoogleGenerativeAI(model=m, google_api_key=api_key)
        response=llm.invoke("Hi")
        print(response.content)
        print(f'{m} model working')
        working_models.append(m)
    except Exception as e:
        error=str(e).split('.')[0]
        print(f'Failed {m} model ({error}')

### Parsing the PDF
We will use pypdf from langchain_community.document_loaders because pypdf or pdfplumber libraries will give us the pdf as raw text while this will use pypdf and give us a document object, so we wont have to loop and convert each page into a document.

In [9]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

pdf=r"C:\Users\Lenovo\OneDrive\Desktop\RAG Project\test.pdf"
loader=PyPDFLoader(pdf)
pages=loader.load()

print(f'{pdf} has been loaded. It has {len(pages)} pages')

C:\Users\Lenovo\OneDrive\Desktop\RAG Project\test.pdf has been loaded. It has 32 pages


Choosing a chunk size of 1000 as it would make rough 150-200 words.

In [10]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks= text_splitter.split_documents(pages)

print(f"The pdf has been split into {len(chunks)} chunks")
print(chunks[21].page_content)

The pdf has been split into 32 chunks
S e a r ch
•Conditions for optimality:
Consistency (or monotonicity): the estimated cost cannot increase
• Most widely know form of best-first search, pronounced A-star search 
• It combines the cost to get to a node and the estimated cost to get to the goal 
• Thus, f(n) = g(n) + h(n)
with g(n) cost from initial state to current node and h(h) estimated cost from current 
state to goal f(n) estimated total cost from start to solution
• The tree-search version of A∗ is optimal if h(n) is admissible, while the graph-search 
version is optimal if h(n) is consistent. 
h(n) must be an admissible heuristic, i.e., it never overestimates the cost
A* search
1525COP506 – Artificial Intelligence – Y . Xie


In [11]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

In [12]:
import google.generativeai as genai

genai.configure(api_key=api_key)

print("Embedding Models:")
for m in genai.list_models():
    if 'embedContent' in m.supported_generation_methods:
        print(f" - {m.name}")

Embedding Models:
 - models/gemini-embedding-001


In [13]:
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=api_key)
test_vector=embeddings.embed_query('Hello world')
print('Embedding created')
print(f'shape of vector: {len(test_vector)}')
print(f'first 5 numbers: {test_vector[:5]}')


Embedding created
shape of vector: 3072
first 5 numbers: [-0.02342152, 0.01676572, 0.009261323, -0.06383, -0.0026262768]


In [14]:
from langchain_community.vectorstores import FAISS

vector_store = FAISS.from_documents(chunks, embeddings)
print('Vector store created')
vector_store.save_local('faiss_index')


Vector store created


In [15]:
query="what is the main topic of this document?"
docs=vector_store.similarity_search(query,k=3)
print(f"Question: {query}")
print(f"Answer: {len(docs)} relevant docs")
for i, doc in enumerate(docs):
    print(f"Result {i+1}:\n")
    print(doc.page_content[:200] + "...")
    print(f"[Source: Page {doc.metadata.get('page', 'Unknown')}]\n")

Question: what is the main topic of this document?
Answer: 3 relevant docs
Result 1:

S e a r ch Ti tl e  o f  th e  
l e ctu r e
• Static agents versus learning agents. 
• An agent might be born with all the 
‘instructions or information about how to 
act in all situations.
• Alternat...
[Source: Page 2]

Result 2:

S e a r ch Ti tl e  o f  th e  
l e ctu r e
A simple example: traveling on a graph
6
A
B
D E
C
F3
3 4
9
4
2
goal state
start state
2
A
B
D
C
F3
3
9
2
goal state
start state
2
25COP506 – Artificial Int...
[Source: Page 5]

Result 3:

25COP506 Artificial 
Intelligence
1
Search algorithms
Yue Xie
25COP506 – Artificial Intelligence – Y . Xie...
[Source: Page 0]



We are now connecting the vector store with the llm to use the data of pdf to answer questions

In [16]:
from langchain_classic.chains import RetrievalQA

llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)
retriever = vector_store.as_retriever(search_kwargs={"k":3})
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True,
    verbose=True
)

print("RAG Chain is ready. The AI can now read your PDF.")

RAG Chain is ready. The AI can now read your PDF.


### Testing

In [17]:
query = "Who is better than Glip Glorp the Martian?"  
print("Thinking...")
result = qa_chain.invoke({"query": query})
print("\nAnswer:")
print(result["result"])
print("\nSources:")
for doc in result["source_documents"]:
    print(f"- Page {doc.metadata.get('page', '?')}")

Thinking...


> Entering new RetrievalQA chain...

> Finished chain.

Answer:
I'm sorry, but the provided text does not contain any information about "Glip Glorp the Martian." Therefore, I cannot answer your question.

Sources:
- Page 21
- Page 19
- Page 20


In [ ]:
import sys

print("PDF chatbot is ready! Type 'exit' or'quit' to stop.")
while True:
    user_input=input("\n You: ")
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Bot: Thank you. Have a nice day!")
        break
    if not user_input.strip():
        continue
    print("Bot is thinking...", end="\r")
    try:
        response = qa_chain.invoke({"query": user_input})
        print(f"Bot: {response['result']}")

    except Exception as e:


        print("Error: {e}")

PDF chatbot is ready! Type 'exit' or'quit' to stop.


In [1]:
!pip install streamlit

In [ ]:
import streamlit as st
import os
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_classic.chains import RetrievalQA  # Using your working library
from dotenv import load_dotenv

# 1. Page Config
st.set_page_config(page_title="My PDF Chatbot", page_icon="🤖")
st.title("🤖 Chat with your PDF")

# 2. Load API Key
try:
    api_key = st.secrets["GOOGLE_API_KEY"]
except FileNotFoundError:
    st.error("Secrets file not found. Please create .streamlit/secrets.toml")
    st.stop()

if not api_key:
    st.error("API Key not found! Please make sure .env file is present.")
    st.stop()

# 3. Setup the AI (Cached so it doesn't reload every time)
@st.cache_resource
def load_chain():
    # A. Load the Embeddings
    embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004", google_api_key=api_key)
    
    # B. Load the Vector Store (We read the folder you already created!)
    # Make sure "faiss_index" folder exists in the same directory
    vector_store = FAISS.load_local("faiss_index", embeddings, allow_dangerous_deserialization=True)
    
    # C. Setup the Brain
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", google_api_key=api_key)
    
    # D. Build the Chain
    qa_chain = RetrievalQA.from_chain_type(
        llm=llm,
        chain_type="stuff",
        retriever=vector_store.as_retriever(search_kwargs={"k": 3}),
        return_source_documents=True
    )
    return qa_chain

# Show a loading spinner while the AI wakes up
with st.spinner("Waking up the AI..."):
    qa_chain = load_chain()

# 4. Chat Interface
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for message in st.session_state.messages:
    with st.chat_message(message["role"]):
        st.markdown(message["content"])

# User Input
if prompt := st.chat_input("Ask a question about your PDF..."):
    # Add user message to history
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.markdown(prompt)

    # Generate Answer
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            response = qa_chain.invoke({"query": prompt})
            answer = response['result']
            st.markdown(answer)
            
            # Show sources in an expander
            with st.expander("View Source Documents"):
                for doc in response['source_documents']:
                    st.write(f"**Page {doc.metadata.get('page', '?')}**")
                    st.text(doc.page_content[:200] + "...")

    # Add bot message to history
    st.session_state.messages.append({"role": "assistant", "content": answer})

2026-02-12 14:37:11.593 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-12 14:37:11.595 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-12 14:37:12.537 
  command:

    streamlit run c:\Users\Lenovo\anaconda3\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-02-12 14:37:12.538 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-12 14:37:12.540 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-12 14:37:12.541 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-02-12 14:37:12.542 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when run

ValidationError: 1 validation error for GoogleGenerativeAIEmbeddings
  Value error, API key required for Gemini Developer API. Provide api_key parameter or set GOOGLE_API_KEY/GEMINI_API_KEY environment variable. [type=value_error, input_value={'model': 'models/text-em... 'google_api_key': None}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.10/v/value_error

In [3]:
!pip install streamlit